In [246]:
import re
import unicodedata
import pandas as pd

In [247]:
df = pd.read_csv(r"dataset\filtering\news_balanced.csv")
print(f'''
      Shape: {df.shape}
      Columns: {df.columns.tolist()}
''')
df.head()


      Shape: (100, 6)
      Columns: ['content_id', 'text', 'section', 'published_date', 'category', 'keywords']



,content_id,text,section,published_date,category,keywords
0,1718879,With artificial intelligence (AI) in the mix a...,Tech,2025-09-29 00:00:00,5G,"5G,Internet,Technology,AI,Smart cities"
1,1442420,"CUPERTINO: Until now, the AirPods Pro were all...",Tech,2024-09-17 00:00:00,Gadgets,"Gadgets,Technology"
2,1297028,"WATERTOWN, New York: A Watertown man was arres...",Tech,2024-03-04 00:00:00,Gadgets,"Gadgets,Courts Crime"
3,1782746,I knew my efforts to learn Japanese before my ...,Tech,2025-12-30 00:00:00,Gadgets,"Gadgets,Technology"
4,1345607,LONDON: British fire chiefs and recycling camp...,Tech,2024-05-13 00:00:00,Gadgets,"Gadgets,Environment"


In [248]:
def clean_text(text):
    text = unicodedata.normalize("NFKC", text)
    
    text = text.replace('\xad', '')
    
    text = re.sub(r'http\S+|www\.\S+', '', text)
    
    text = re.sub(r'<[^>]+>', '', text)
    
    text = ''.join(ch for ch in text if unicodedata.category(ch)[0] != "C")
    
    text = re.sub(r'([.!?,])([A-Z])', r'\1 \2', text)
    
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()

In [249]:
df['clean_text'] = df['text'].apply(clean_text)

In [250]:
df['published_date'] = pd.to_datetime(df['published_date'])

In [251]:
df.clean_text[4]

'LONDON: British fire chiefs and recycling campaigners have warned that fires caused by discarded batteries in electricals are on the rise, causing damage and spikes in air pollution levels. According to a study by Material Focus, which leads the Recycle Your Electricals campaign, battery fires in bin lorries and at waste sites have risen by more than 70% since 2022, with more than 1,200 estimated to have occurred last year. It said that the steep rise in the number of portable electrical items containing lithium-ion batteries being bought and used by the public was leading to an increased risk of fires, with the study indicating that 1.6 billion batteries were thrown away last year, including 1.1 billion containing lithium-ion batteries. Found inside most common portable electronics, including laptops, phones, tablets, earpods and vapes, lithium-ion batteries can be crushed or damaged if not recycled and instead end up in bin lorries or waste sites – which can lead to fires. Alongside

In [252]:
clean_text(df.clean_text[4])

'LONDON: British fire chiefs and recycling campaigners have warned that fires caused by discarded batteries in electricals are on the rise, causing damage and spikes in air pollution levels. According to a study by Material Focus, which leads the Recycle Your Electricals campaign, battery fires in bin lorries and at waste sites have risen by more than 70% since 2022, with more than 1,200 estimated to have occurred last year. It said that the steep rise in the number of portable electrical items containing lithium-ion batteries being bought and used by the public was leading to an increased risk of fires, with the study indicating that 1.6 billion batteries were thrown away last year, including 1.1 billion containing lithium-ion batteries. Found inside most common portable electronics, including laptops, phones, tablets, earpods and vapes, lithium-ion batteries can be crushed or damaged if not recycled and instead end up in bin lorries or waste sites – which can lead to fires. Alongside

In [253]:
df.head()

,content_id,text,section,published_date,category,keywords,clean_text
0,1718879,With artificial intelligence (AI) in the mix a...,Tech,2025-09-29,5G,"5G,Internet,Technology,AI,Smart cities",With artificial intelligence (AI) in the mix a...
1,1442420,"CUPERTINO: Until now, the AirPods Pro were all...",Tech,2024-09-17,Gadgets,"Gadgets,Technology","CUPERTINO: Until now, the AirPods Pro were all..."
2,1297028,"WATERTOWN, New York: A Watertown man was arres...",Tech,2024-03-04,Gadgets,"Gadgets,Courts Crime","WATERTOWN, New York: A Watertown man was arres..."
3,1782746,I knew my efforts to learn Japanese before my ...,Tech,2025-12-30,Gadgets,"Gadgets,Technology",I knew my efforts to learn Japanese before my ...
4,1345607,LONDON: British fire chiefs and recycling camp...,Tech,2024-05-13,Gadgets,"Gadgets,Environment",LONDON: British fire chiefs and recycling camp...


In [254]:
df['timestamp'] = df['published_date'].astype('int64') // 10**9

In [255]:
df['year'] = df['published_date'].dt.year
df['month'] = df['published_date'].dt.month
df['week'] = df['published_date'].dt.isocalendar().week

In [256]:
import nltk
nltk.download('punkt')
from nltk.tokenize import sent_tokenize

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\gaura\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [257]:
df['sentences'] = df['clean_text'].apply(sent_tokenize)

In [258]:
df_sent = df.explode('sentences').reset_index(drop=True)
df_sent.rename(columns={'sentences': 'sentence'}, inplace=True)

In [259]:
print(df_sent.shape)

category_counts = df_sent['category'].value_counts()
category_counts

(2968, 12)


AI         1447
Gadgets    1111
5G          410
Name: category, dtype: int64

In [260]:
df_sent = df_sent[df_sent['sentence'].str.len() > 40]

In [261]:
print(df_sent.shape)

category_counts = df_sent['category'].value_counts()
category_counts

(2786, 12)


AI         1360
Gadgets    1040
5G          386
Name: category, dtype: int64

In [262]:
df_final = df_sent[['content_id', 'sentence', 'published_date', 'timestamp', 'category']]

print(df_final.shape)
df_final.head()

(2786, 5)


,content_id,sentence,published_date,timestamp,category
0,1718879,With artificial intelligence (AI) in the mix a...,2025-09-29,1759104000,5G
1,1718879,"Take Putrajaya Corporation, for instance, whic...",2025-09-29,1759104000,5G
2,1718879,"At its SCEKL booth, the city featured everythi...",2025-09-29,1759104000,5G
3,1718879,"In the case of smart traffic lights, AI stream...",2025-09-29,1759104000,5G
4,1718879,The system achieves this by having CCTV camera...,2025-09-29,1759104000,5G


In [263]:
df_final.sentence[37]

'The company is also planning to integrate a hearing test into the AirPods, whereby the user taps the screen when they hear a certain frequency.'

In [264]:
df_final.to_csv(r"dataset\preprocessing\prePro-news_balanced.csv", index=False)